# Step 6.3

In [82]:
import os

base = r"C:\Users\deep8\breast_cancer_project_folder"
os.listdir(base)


['.ipynb_checkpoints',
 'Census_allWed Jan 28 09_32_24 2026.csv',
 'chembl_mechanism.json',
 'chembl_target.json',
 'chembl_uniprot_mapping.txt',
 'clinical.tsv',
 'CRISPR_gene_effect.csv',
 'DEGs_HER2_vs_Normal_FDR0.05_log2FC1.csv',
 'DEGs_LumA_vs_Normal_FDR0.05_log2FC1.csv',
 'DEGs_LumB_vs_Normal_FDR0.05_log2FC1.csv',
 'DEGs_TNBC_vs_Normal_FDR0.05_log2FC1.csv',
 'Drug_Targets_PPI_Filtered.csv',
 'exposure.tsv',
 'family_history.tsv',
 'follow_up.tsv',
 'gdc_manifest.2026-01-23.234051.txt',
 'Heatmap_Top50_LumA_vs_Normal_ORDERED_600dpi.png',
 'HER2_PPI_Pagerank.csv',
 'LumA_PPI_Pagerank.csv',
 'LumB_PPI_Pagerank.csv',
 'MA_HER2_vs_Normal_600dpi.png',
 'MA_LumA_vs_Normal_600dpi.png',
 'MA_LumB_vs_Normal_600dpi.png',
 'MA_TNBC_vs_Normal_600dpi.png',
 'metadata.cart.2026-01-27.json',
 'metadata.json',
 'pathology_detail.tsv',
 'PPI_TCGA_BRCA_STRING_HQ.csv',
 'Seeds_HER2.csv',
 'Seeds_LumA.csv',
 'Seeds_LumB.csv',
 'Seeds_TNBC.csv',
 'TCGA.BRCA (2).sampleMap_BRCA_clinicalMatrix',
 'TCGA.B

In [84]:
import pandas as pd
import networkx as nx

base = r"C:\Users\deep8\breast_cancer_project_folder"

# load PPI edge list
ppi = pd.read_csv(f"{base}/PPI_TCGA_BRCA_STRING_HQ.csv")

ppi.head(), ppi.shape


(               protein1              protein2  neighborhood  fusion  \
 0  9606.ENSP00000000233  9606.ENSP00000262812             0       0   
 1  9606.ENSP00000000233  9606.ENSP00000158762             0       0   
 2  9606.ENSP00000000233  9606.ENSP00000480707             0       0   
 3  9606.ENSP00000000233  9606.ENSP00000263245             0       0   
 4  9606.ENSP00000000233  9606.ENSP00000484121             0       0   
 
    cooccurence  coexpression  experimental  database  textmining  \
 0            0           190           163       600         173   
 1            0             0           147         0         736   
 2            0            98           187       600         194   
 3            0            63           391       600         527   
 4            0             0           519         0         566   
 
    combined_score gene1    gene2  
 0             745  ARF5     COPE  
 1             765  ARF5    ACAP1  
 2             731  ARF5    COPZ2  
 3    

In [86]:
# build undirected PPI graph
G = nx.from_pandas_edgelist(
    ppi,
    source="gene1",
    target="gene2"
)

# basic sanity checks
G.number_of_nodes(), G.number_of_edges()


(10095, 97248)

In [88]:
# extract giant connected component
G_cc = G.subgraph(max(nx.connected_components(G), key=len)).copy()

# check size
G_cc.number_of_nodes(), G_cc.number_of_edges()


(9875, 97117)

In [90]:
# load drug–target mapping (already filtered to PPI nodes)
drug_targets = pd.read_csv(
    f"{base}/Drug_Targets_PPI_Filtered.csv"
)

drug_targets.head(), drug_targets.shape


(  target_chembl_id    targets targets_ppi
 0       CHEMBL1778  ['IL2RA']   ['IL2RA']
 1       CHEMBL1782   ['FDPS']    ['FDPS']
 2       CHEMBL1783  ['VEGFA']   ['VEGFA']
 3       CHEMBL1785  ['EDNRB']   ['EDNRB']
 4       CHEMBL1786  ['IMPA1']   ['IMPA1'],
 (522, 3))

In [92]:
# load HER2 seeds
her2_seeds = pd.read_csv(
    f"{base}/Seeds_HER2.csv"
)["gene"].tolist()

len(her2_seeds)


3883

In [96]:
import numpy as np

# copy graph to avoid modifying original
G_tmp = G_cc.copy()

# add a virtual super-source node
SUPER_SOURCE = "__HER2_SEED_SOURCE__"
G_tmp.add_node(SUPER_SOURCE)

# connect super-source to all HER2 seeds
for g in her2_seeds:
    if g in G_tmp:
        G_tmp.add_edge(SUPER_SOURCE, g)

# compute shortest paths from super-source
dist_to_her2 = nx.single_source_shortest_path_length(G_tmp, SUPER_SOURCE)

# compute drug → HER2 proximity
rows = []
for _, row in drug_targets.iterrows():
    drug = row["target_chembl_id"]
    targets = eval(row["targets_ppi"])
    dists = [dist_to_her2[t] - 1 for t in targets if t in dist_to_her2]
    min_dist = np.min(dists) if len(dists) > 0 else np.nan
    rows.append((drug, min_dist))

drug_her2_shortest = pd.DataFrame(
    rows, columns=["drug_chembl_id", "min_shortest_path_to_HER2"]
)

drug_her2_shortest.head(), drug_her2_shortest.shape


(  drug_chembl_id  min_shortest_path_to_HER2
 0     CHEMBL1778                          0
 1     CHEMBL1782                          0
 2     CHEMBL1783                          0
 3     CHEMBL1785                          0
 4     CHEMBL1786                          1,
 (522, 2))

In [98]:
drug_her2_shortest["min_shortest_path_to_HER2"].value_counts().head()


min_shortest_path_to_HER2
0    269
1    251
2      2
Name: count, dtype: int64

In [100]:
drug_her2_shortest["min_shortest_path_to_HER2"].describe()


count    522.000000
mean       0.488506
std        0.507962
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        2.000000
Name: min_shortest_path_to_HER2, dtype: float64

In [102]:
drug_her2_shortest.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_HER2_MinShortestPath.csv",
    index=False
)
print("✅ Saved: Drug_HER2_MinShortestPath.csv")


✅ Saved: Drug_HER2_MinShortestPath.csv


# Luminal A

In [123]:
import pandas as pd

drug_targets_ppi = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Targets_PPI_Filtered.csv"
)

drug_targets_ppi.head(), drug_targets_ppi.shape


(  target_chembl_id    targets targets_ppi
 0       CHEMBL1778  ['IL2RA']   ['IL2RA']
 1       CHEMBL1782   ['FDPS']    ['FDPS']
 2       CHEMBL1783  ['VEGFA']   ['VEGFA']
 3       CHEMBL1785  ['EDNRB']   ['EDNRB']
 4       CHEMBL1786  ['IMPA1']   ['IMPA1'],
 (522, 3))

In [125]:
import numpy as np
import networkx as nx

rows = []

for _, row in drug_targets_ppi.iterrows():
    targets = eval(row["targets_ppi"]) if isinstance(row["targets_ppi"], str) else row["targets_ppi"]
    dists = [dist_to_luma[t] for t in targets if t in dist_to_luma]

    rows.append({
        "drug_chembl_id": row["target_chembl_id"],
        "min_shortest_path_to_LumA": min(dists) if len(dists) > 0 else np.nan
    })

drug_luma_shortest = pd.DataFrame(rows)

drug_luma_shortest.head(), drug_luma_shortest.shape


(  drug_chembl_id  min_shortest_path_to_LumA
 0     CHEMBL1778                          1
 1     CHEMBL1782                          1
 2     CHEMBL1783                          1
 3     CHEMBL1785                          1
 4     CHEMBL1786                          2,
 (522, 2))

In [127]:
drug_luma_shortest.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_LumA_ShortestPath.csv",
    index=False
)

print("Saved: Drug_LumA_ShortestPath.csv")


Saved: Drug_LumA_ShortestPath.csv


In [137]:
# keep only LumB seeds present in PPI graph
lumb_seeds_ppi = set(lumb_seeds).intersection(G_cc.nodes())

len(lumb_seeds), len(lumb_seeds_ppi)


(3901, 3841)

In [141]:
# compute distance to LumB seeds (SAFE version)
dist_to_lumb = {}

for seed in lumb_seeds_ppi:
    lengths = nx.single_source_shortest_path_length(G_cc, seed)
    for node, d in lengths.items():
        if node not in dist_to_lumb or d < dist_to_lumb[node]:
            dist_to_lumb[node] = d

len(dist_to_lumb)


9875

In [145]:
import ast

rows = []

for _, row in drug_targets_ppi.iterrows():
    targets = ast.literal_eval(row["targets_ppi"])  # FIX
    dists = [dist_to_lumb[t] for t in targets if t in dist_to_lumb]

    rows.append({
        "drug_chembl_id": row["target_chembl_id"],
        "min_shortest_path_to_LumB": min(dists) if len(dists) > 0 else np.nan
    })

drug_lumb_shortest = pd.DataFrame(rows)

drug_lumb_shortest.head(), drug_lumb_shortest.shape


(  drug_chembl_id  min_shortest_path_to_LumB
 0     CHEMBL1778                          1
 1     CHEMBL1782                          1
 2     CHEMBL1783                          1
 3     CHEMBL1785                          0
 4     CHEMBL1786                          1,
 (522, 2))

In [147]:
drug_lumb_shortest["min_shortest_path_to_LumB"].value_counts(dropna=False)


min_shortest_path_to_LumB
1    263
0    257
2      2
Name: count, dtype: int64

In [149]:
drug_lumb_shortest.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_LumB_ShortestPath.csv",
    index=False
)


# TNBC shortest-path proximity

In [152]:
# load TNBC seeds (already filtered to PPI earlier)
tnbc_seeds = set(pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Seeds_TNBC.csv"
)["gene"])

# keep only seeds present in PPI graph
tnbc_seeds_ppi = tnbc_seeds.intersection(G_cc.nodes())

len(tnbc_seeds), len(tnbc_seeds_ppi)


(3993, 3929)

In [154]:
# build TNBC distance map (safe aggregation)
dist_to_tnbc = {}

for seed in tnbc_seeds_ppi:
    lengths = nx.single_source_shortest_path_length(G_cc, seed)
    for node, d in lengths.items():
        if (node not in dist_to_tnbc) or (d < dist_to_tnbc[node]):
            dist_to_tnbc[node] = d

len(dist_to_tnbc)


9875

In [158]:
import numpy as np

rows = []

for _, row in drug_targets_ppi.iterrows():
    # FIX: convert string → list
    targets = eval(row["targets_ppi"]) if isinstance(row["targets_ppi"], str) else row["targets_ppi"]

    dists = [dist_to_tnbc[t] for t in targets if t in dist_to_tnbc]

    rows.append({
        "drug_chembl_id": row["target_chembl_id"],
        "min_shortest_path_to_TNBC": min(dists) if len(dists) > 0 else np.nan
    })

drug_tnbc_shortest = pd.DataFrame(rows)

drug_tnbc_shortest.head(), drug_tnbc_shortest.shape


(  drug_chembl_id  min_shortest_path_to_TNBC
 0     CHEMBL1778                          0
 1     CHEMBL1782                          1
 2     CHEMBL1783                          0
 3     CHEMBL1785                          0
 4     CHEMBL1786                          1,
 (522, 2))

# PHASE 6.3.2 — Random-walk

In [161]:
import pandas as pd
import numpy as np

# load inputs
pr_her2 = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\HER2_PPI_Pagerank.csv",
    index_col=0
)
drug_targets_ppi = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Targets_PPI_Filtered.csv"
)

rows = []
for _, row in drug_targets_ppi.iterrows():
    targets = row["targets_ppi"]
    if isinstance(targets, str):
        targets = eval(targets)

    scores = [pr_her2.loc[t, "pagerank"] for t in targets if t in pr_her2.index]

    rows.append({
        "drug_chembl_id": row["target_chembl_id"],
        "rw_mean_HER2": np.mean(scores) if scores else np.nan,
        "rw_max_HER2":  np.max(scores) if scores else np.nan
    })

drug_her2_rw = pd.DataFrame(rows)
drug_her2_rw.head(), drug_her2_rw.shape


(  drug_chembl_id  rw_mean_HER2  rw_max_HER2
 0     CHEMBL1778      0.000275     0.000275
 1     CHEMBL1782      0.000143     0.000143
 2     CHEMBL1783      0.000742     0.000742
 3     CHEMBL1785      0.000159     0.000159
 4     CHEMBL1786      0.000037     0.000037,
 (522, 3))

In [165]:
# save HER2 random-walk proximity features
drug_her2_rw.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_RW_Proximity_HER2.csv",
    index=False
)

print("✅ Saved: Drug_RW_Proximity_HER2.csv")


✅ Saved: Drug_RW_Proximity_HER2.csv


In [169]:
import pandas as pd
import numpy as np

# load LumA PageRank scores
pr_luma = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LumA_PPI_Pagerank.csv",
    index_col=0
)

pr_luma.head(), pr_luma.shape



(       pagerank
 TP53   0.002233
 SRC    0.002190
 EP300  0.001869
 EGFR   0.001740
 HRAS   0.001672,
 (9875, 1))

In [183]:
drug_targets_ppi["targets_ppi"] = drug_targets_ppi["targets_ppi"].apply(
    lambda x: eval(x) if isinstance(x, str) else x
)


In [185]:
rows = []

for _, row in drug_targets_ppi.iterrows():
    targets = row["targets_ppi"]
    scores = [pr_luma.loc[t, "pagerank"] for t in targets if t in pr_luma.index]

    rows.append({
        "drug_chembl_id": row["target_chembl_id"],
        "rw_mean_LumA": np.mean(scores) if len(scores) > 0 else np.nan,
        "rw_max_LumA": np.max(scores) if len(scores) > 0 else np.nan
    })

drug_luma_rw = pd.DataFrame(rows)

drug_luma_rw.head(), drug_luma_rw.isna().sum()


(  drug_chembl_id  rw_mean_LumA  rw_max_LumA
 0     CHEMBL1778      0.000302     0.000302
 1     CHEMBL1782      0.000091     0.000091
 2     CHEMBL1783      0.000768     0.000768
 3     CHEMBL1785      0.000129     0.000129
 4     CHEMBL1786      0.000024     0.000024,
 drug_chembl_id    0
 rw_mean_LumA      0
 rw_max_LumA       0
 dtype: int64)

In [187]:
drug_targets_ppi["n_targets"] = drug_targets_ppi["targets_ppi"].apply(len)
drug_targets_ppi["n_targets"].value_counts().head()


n_targets
1    522
Name: count, dtype: int64

In [189]:
drug_luma_rw.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_LumA_RW_Proximity.csv",
    index=False
)

print("Saved: Drug_LumA_RW_Proximity.csv")


Saved: Drug_LumA_RW_Proximity.csv


In [191]:
# load LumB PageRank
pr_lumb = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LumB_PPI_Pagerank.csv",
    index_col=0
)["pagerank"]

rows = []

for _, row in drug_targets_ppi.iterrows():
    targets = row["targets_ppi"]
    scores = [pr_lumb[t] for t in targets if t in pr_lumb.index]

    rows.append({
        "drug_chembl_id": row["target_chembl_id"],
        "rw_mean_LumB": np.mean(scores) if len(scores) > 0 else np.nan,
        "rw_max_LumB": np.max(scores) if len(scores) > 0 else np.nan
    })

drug_lumb_rw = pd.DataFrame(rows)

drug_lumb_rw.head(), drug_lumb_rw.isna().sum()


(  drug_chembl_id  rw_mean_LumB  rw_max_LumB
 0     CHEMBL1778      0.000240     0.000240
 1     CHEMBL1782      0.000090     0.000090
 2     CHEMBL1783      0.000708     0.000708
 3     CHEMBL1785      0.000146     0.000146
 4     CHEMBL1786      0.000036     0.000036,
 drug_chembl_id    0
 rw_mean_LumB      0
 rw_max_LumB       0
 dtype: int64)

In [193]:
drug_lumb_rw.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_LumB_RW_Proximity.csv",
    index=False
)

print("Saved: Drug_LumB_RW_Proximity.csv")


Saved: Drug_LumB_RW_Proximity.csv


In [195]:
# load TNBC PageRank
pr_tnbc = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\TNBC_PPI_Pagerank.csv",
    index_col=0
)["pagerank"]

rows = []

for _, row in drug_targets_ppi.iterrows():
    targets = row["targets_ppi"]
    scores = [pr_tnbc[t] for t in targets if t in pr_tnbc.index]

    rows.append({
        "drug_chembl_id": row["target_chembl_id"],
        "rw_mean_TNBC": np.mean(scores) if len(scores) > 0 else np.nan,
        "rw_max_TNBC": np.max(scores) if len(scores) > 0 else np.nan
    })

drug_tnbc_rw = pd.DataFrame(rows)

drug_tnbc_rw.head(), drug_tnbc_rw.isna().sum()


(  drug_chembl_id  rw_mean_TNBC  rw_max_TNBC
 0     CHEMBL1778      0.000271     0.000271
 1     CHEMBL1782      0.000099     0.000099
 2     CHEMBL1783      0.000735     0.000735
 3     CHEMBL1785      0.000167     0.000167
 4     CHEMBL1786      0.000048     0.000048,
 drug_chembl_id    0
 rw_mean_TNBC      0
 rw_max_TNBC       0
 dtype: int64)

In [197]:
drug_tnbc_rw.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_TNBC_RW_Features.csv",
    index=False
)


# 6.3.3 — Propagation score overlap

In [200]:
import pandas as pd

# load HER2 PageRank
pr_her2 = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\HER2_PPI_Pagerank.csv",
    index_col=0
)

pr_her2.head(), pr_her2.shape


(        pagerank
 SRC     0.001975
 TP53    0.001818
 EGFR    0.001521
 RPS27A  0.001392
 EP300   0.001352,
 (9875, 1))

In [202]:
import numpy as np

rows = []

for _, row in drug_targets_ppi.iterrows():
    targets = row["targets_ppi"]
    
    # keep only targets present in PageRank index
    scores = [
        pr_her2.loc[t, "pagerank"]
        for t in targets
        if t in pr_her2.index
    ]
    
    rows.append({
        "drug_chembl_id": row["target_chembl_id"],
        "prop_mean_HER2": np.mean(scores) if len(scores) > 0 else np.nan,
        "prop_max_HER2": np.max(scores) if len(scores) > 0 else np.nan
    })

drug_her2_prop = pd.DataFrame(rows)

drug_her2_prop.head(), drug_her2_prop.isna().sum()


(  drug_chembl_id  prop_mean_HER2  prop_max_HER2
 0     CHEMBL1778        0.000275       0.000275
 1     CHEMBL1782        0.000143       0.000143
 2     CHEMBL1783        0.000742       0.000742
 3     CHEMBL1785        0.000159       0.000159
 4     CHEMBL1786        0.000037       0.000037,
 drug_chembl_id    0
 prop_mean_HER2    0
 prop_max_HER2     0
 dtype: int64)

In [204]:
drug_her2_prop.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_HER2_Propagation_Overlap.csv",
    index=False
)

print("Saved: Drug_HER2_Propagation_Overlap.csv")


Saved: Drug_HER2_Propagation_Overlap.csv


In [206]:
# load LumA PageRank
pr_luma = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LumA_PPI_Pagerank.csv",
    index_col=0
)

pr_luma.head(), pr_luma.shape


(       pagerank
 TP53   0.002233
 SRC    0.002190
 EP300  0.001869
 EGFR   0.001740
 HRAS   0.001672,
 (9875, 1))

In [208]:
import numpy as np

rows = []

for _, row in drug_targets_ppi.iterrows():
    targets = row["targets_ppi"]

    scores = [
        pr_luma.loc[t, "pagerank"]
        for t in targets
        if t in pr_luma.index
    ]

    rows.append({
        "drug_chembl_id": row["target_chembl_id"],
        "prop_mean_LumA": np.mean(scores) if len(scores) > 0 else np.nan,
        "prop_max_LumA": np.max(scores) if len(scores) > 0 else np.nan
    })

drug_luma_prop = pd.DataFrame(rows)

drug_luma_prop.head(), drug_luma_prop.isna().sum()


(  drug_chembl_id  prop_mean_LumA  prop_max_LumA
 0     CHEMBL1778        0.000302       0.000302
 1     CHEMBL1782        0.000091       0.000091
 2     CHEMBL1783        0.000768       0.000768
 3     CHEMBL1785        0.000129       0.000129
 4     CHEMBL1786        0.000024       0.000024,
 drug_chembl_id    0
 prop_mean_LumA    0
 prop_max_LumA     0
 dtype: int64)

In [210]:
drug_luma_prop.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_LumA_Propagation_Overlap.csv",
    index=False
)

print("Saved: Drug_LumA_Propagation_Overlap.csv")


Saved: Drug_LumA_Propagation_Overlap.csv


In [212]:
# load LumB PageRank
pr_lumb = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LumB_PPI_Pagerank.csv",
    index_col=0
)

pr_lumb.head(), pr_lumb.shape


(        pagerank
 SRC     0.001973
 TP53    0.001848
 EGFR    0.001528
 EP300   0.001395
 RPS27A  0.001390,
 (9875, 1))

In [214]:
import numpy as np

rows = []

for _, row in drug_targets_ppi.iterrows():
    targets = row["targets_ppi"]

    scores = [
        pr_lumb.loc[t, "pagerank"]
        for t in targets
        if t in pr_lumb.index
    ]

    rows.append({
        "drug_chembl_id": row["target_chembl_id"],
        "prop_mean_LumB": np.mean(scores) if len(scores) > 0 else np.nan,
        "prop_max_LumB": np.max(scores) if len(scores) > 0 else np.nan
    })

drug_lumb_prop = pd.DataFrame(rows)

drug_lumb_prop.head(), drug_lumb_prop.isna().sum()


(  drug_chembl_id  prop_mean_LumB  prop_max_LumB
 0     CHEMBL1778        0.000240       0.000240
 1     CHEMBL1782        0.000090       0.000090
 2     CHEMBL1783        0.000708       0.000708
 3     CHEMBL1785        0.000146       0.000146
 4     CHEMBL1786        0.000036       0.000036,
 drug_chembl_id    0
 prop_mean_LumB    0
 prop_max_LumB     0
 dtype: int64)

In [216]:
drug_lumb_prop.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_LumB_Propagation_Overlap.csv",
    index=False
)

print("Saved: Drug_LumB_Propagation_Overlap.csv")


Saved: Drug_LumB_Propagation_Overlap.csv


In [218]:
# load TNBC PageRank
pr_tnbc = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\TNBC_PPI_Pagerank.csv",
    index_col=0
)

pr_tnbc.head(), pr_tnbc.shape


(        pagerank
 SRC     0.001931
 TP53    0.001838
 EGFR    0.001459
 RPS27A  0.001389
 EP300   0.001337,
 (9875, 1))

In [220]:
import numpy as np
import pandas as pd

rows = []

for _, row in drug_targets_ppi.iterrows():
    targets = row["targets_ppi"]

    scores = [
        pr_tnbc.loc[t, "pagerank"]
        for t in targets
        if t in pr_tnbc.index
    ]

    rows.append({
        "drug_chembl_id": row["target_chembl_id"],
        "prop_mean_TNBC": np.mean(scores) if len(scores) > 0 else np.nan,
        "prop_max_TNBC": np.max(scores) if len(scores) > 0 else np.nan
    })

drug_tnbc_prop = pd.DataFrame(rows)

drug_tnbc_prop.head(), drug_tnbc_prop.isna().sum()


(  drug_chembl_id  prop_mean_TNBC  prop_max_TNBC
 0     CHEMBL1778        0.000271       0.000271
 1     CHEMBL1782        0.000099       0.000099
 2     CHEMBL1783        0.000735       0.000735
 3     CHEMBL1785        0.000167       0.000167
 4     CHEMBL1786        0.000048       0.000048,
 drug_chembl_id    0
 prop_mean_TNBC    0
 prop_max_TNBC     0
 dtype: int64)

In [222]:
drug_tnbc_prop.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_TNBC_Propagation.csv",
    index=False
)
